# O3a training → O4a event detection

This separate notebook builds a balanced binary dataset: **1 = simulated gravitational-wave event present**, **0 = noise only**. O3a supplies training and validation; O4a is reserved for final testing. It reads existing local checkpoints and never downloads data or changes the existing pipeline.

A noise segment is an array of strain samples. A simulated waveform is another array of strain samples. Positive data is created as `noise + waveform`: values at matching sample positions are added. Negative data is just `noise`, with no injected waveform. Both classes are whitened and bandpassed. Each final example has shape `(16384,)`, or four seconds at 4096 samples/second.

Run the cells from top to bottom with NumPy, SciPy, and scikit-learn installed. Only load trusted local pickle files. The dataset is saved once to a new path; an existing output is never overwritten.

## 1. Imports, paths, and fixed settings

Run from the project root or its `notebooks` directory. These numerical settings match `config.py` and `ligo_pipeline/preprocessing.py`, but all processing code lives here. The local checkpoints were saved with a NumPy 2 module path; the small compatibility alias allows direct `pickle.load` with the project's NumPy 1.26 environment as well.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pickle
import sys
import warnings

import numpy as np
import scipy
from scipy import signal
import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, roc_auc_score

if np.__version__.split(".")[0] == "1":
    sys.modules.setdefault("numpy._core.numeric", np.core.numeric)

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
positive_path = project_root / "artifacts/data/training_data/training_data.pkl"
noise_paths = {
    "O3a": project_root / "artifacts/data/noise/o3a_noise_segments.pkl",
    "O4a": project_root / "artifacts/data/noise/o4a_noise_segments.pkl",
}
output_path = project_root / "artifacts/classification/o3a_train_o4a_event_detection.pkl"
SAMPLE_RATE = 4096
WINDOW_SAMPLES = 16384
SEED = 20260924

# Check every required noise checkpoint before doing any data processing.
for run, path in noise_paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"Missing required {run} noise checkpoint: {path}. Stopping; no download is attempted.")
if not positive_path.is_file():
    raise FileNotFoundError(f"Missing positive checkpoint: {positive_path}")
if output_path.exists():
    raise FileExistsError(f"Output already exists and will not be overwritten: {output_path}")
print("Local noise files:")
for path in sorted((project_root / "artifacts/data/noise").iterdir()):
    print(path.name, f"{path.stat().st_size:,} bytes")
print("New output:", output_path)

Local noise files:
o3a_noise_segments.pkl 536,871,812 bytes
o3a_noise_segments.pkl.meta.json 313 bytes
o4a_noise_segments.pkl 536,871,812 bytes
o4a_noise_segments.pkl.meta.json 313 bytes
New output: /Users/alilordifar/Library/CloudStorage/GoogleDrive-ali.lordifar@gmail.com/My Drive/Projects/ligo-parameter-inference/artifacts/classification/o3a_train_o4a_event_detection.pkl


## 2. A tiny example of adding arrays

These three samples are only an illustration, not training data. The real positive examples are already generated and processed, so we do not inject or preprocess them again. The `theta` arrays describe physical parameters; they are **not binary classification labels**.

In [2]:
toy_noise = np.array([0.1, -0.2, 0.3])
toy_waveform = np.array([0.0, 0.5, -0.1])
print("noise:            ", toy_noise)
print("waveform:         ", toy_waveform)
print("positive = sum:   ", toy_noise + toy_waveform)
print("negative = noise: ", toy_noise)

noise:             [ 0.1 -0.2  0.3]
waveform:          [ 0.   0.5 -0.1]
positive = sum:    [0.1 0.3 0.2]
negative = noise:  [ 0.1 -0.2  0.3]


## 3. Load the existing positives directly with pickle

Keep the existing split exactly: 750 O3a training positives, 50 O3a validation positives, and 800 O4a final-test positives. Only the three `X` arrays are used.

In [3]:
with positive_path.open("rb") as handle:
    positive_checkpoint = pickle.load(handle)
positive_train = np.asarray(positive_checkpoint["X_train"])
positive_validation = np.asarray(positive_checkpoint["X_test_o3a"])
positive_test = np.asarray(positive_checkpoint["X_test_o4a"])
del positive_checkpoint

for name, values, expected_rows in [
    ("O3a training positives", positive_train, 750),
    ("O3a validation positives", positive_validation, 50),
    ("O4a test positives", positive_test, 800),
]:
    print(name, values.shape, "finite:", np.isfinite(values).all())
    assert values.shape == (expected_rows, WINDOW_SAMPLES)
    assert np.isfinite(values).all()
print("First positive's first five processed samples:", positive_train[0, :5])

O3a training positives (750, 16384) finite: True
O3a validation positives (50, 16384) finite: True
O4a test positives (800, 16384) finite: True
First positive's first five processed samples: [21.27249347 17.49582729 13.92048182 12.03556783 12.82354152]


## 4. Inspect the clean-noise provenance

Each checkpoint is a dictionary: `segment_start_GPS → (strain_array, sample_times_array)`. Each run contains eight 1024-second segments, each with 4,194,304 samples.

The existing acquisition code excludes segments within **3600 seconds of cataloged event times** and checks data quality. The saved acquisition log (`artifacts/logs/pipeline_20260912T000605Z.log`) reports 63/63 resolved event times for O3a and 140/140 for O4a, with zero lookup failures. We reuse these vetted checkpoints and require their sidecar metadata to record that exclusion. All selected windows stay inside those segments. This relies on the catalog and quality vetting at acquisition time; it is not a fresh catalog check or a guarantee that every possible transient is absent. No network request is made.

In [4]:
noise_metadata = {}
for run, path in noise_paths.items():
    sidecar_path = Path(str(path) + ".meta.json")
    if not sidecar_path.is_file():
        raise FileNotFoundError(f"Missing noise provenance metadata: {sidecar_path}")
    with sidecar_path.open() as handle:
        details = json.load(handle)
    assert details["run"] == run
    assert details["detector"] == "H1"
    assert details["sample_rate_hz"] == SAMPLE_RATE
    assert details["segment_duration_sec"] == 1024
    assert details["n_segments"] == 8
    assert details["event_exclusion_buffer_sec"] >= 3600
    noise_metadata[run] = details
    print(run, details)

O3a {'created_utc': '2026-09-12T00:27:30.413127+00:00', 'run': 'O3a', 'detector': 'H1', 'segment_duration_sec': 1024, 'n_segments': 8, 'sample_rate_hz': 4096, 'std_range': [1e-19, 1e-17], 'dq_cbc_bits': [0, 1, 2, 3], 'event_exclusion_buffer_sec': 3600, 'seed': 0}
O4a {'created_utc': '2026-09-12T01:31:04.334223+00:00', 'run': 'O4a', 'detector': 'H1', 'segment_duration_sec': 1024, 'n_segments': 8, 'sample_rate_hz': 4096, 'std_range': [1e-19, 1e-17], 'dq_cbc_bits': [0, 1, 2, 3], 'event_exclusion_buffer_sec': 3600, 'seed': 0}


## 5. Select distinct windows and process noise only

For each run we select **800 different four-second windows**, without replacement, spread equally over the eight segments (100 per segment). Candidate windows lie on a non-overlapping four-second grid, at least 60 seconds from segment boundaries. No selected four-second window is reused, including between O3a training and validation.

A power spectral density (PSD) estimates how noise power varies with frequency. Estimate it from the **full original clean segment**, using Welch's method with four-second subsegments. Reuse that estimate for windows from the same segment.

The existing positive pipeline whitens and filters a surrounding buffer before cropping to four seconds. To follow that approach and reduce filter-edge effects, take 15 seconds on either side of each negative window's center, whiten by dividing its Fourier transform by the square root of the interpolated PSD, apply the same fourth-order Butterworth 35–350 Hz bandpass with `filtfilt`, then keep the central 16384 samples. The selected final window is exactly four seconds; the extra context is discarded. No waveform is generated or added. Context buffers can overlap, but final negative windows do not. Positive injection buffers can be longer for long waveforms; existing positives are kept unchanged.

In [5]:
negative_by_run = {}
negative_records_by_run = {}
segment_ranges_by_run = {}
filter_b, filter_a = signal.butter(4, [35 / (SAMPLE_RATE / 2), 350 / (SAMPLE_RATE / 2)], btype="band")
buffer_half_samples = 15 * SAMPLE_RATE
edge_margin_samples = 60 * SAMPLE_RATE

for run_index, (run, path) in enumerate(noise_paths.items()):
    with path.open("rb") as handle:
        noise_segments = pickle.load(handle)
    assert isinstance(noise_segments, dict) and len(noise_segments) == 8
    rng = np.random.default_rng(SEED + run_index)
    processed_windows = []
    window_records = []
    segment_ranges = []

    for segment_gps, (clean_noise, sample_times) in sorted(noise_segments.items()):
        clean_noise = np.asarray(clean_noise)
        sample_times = np.asarray(sample_times)
        assert clean_noise.shape == sample_times.shape == (1024 * SAMPLE_RATE,)
        assert np.isfinite(clean_noise).all() and np.isfinite(sample_times).all()
        assert np.isclose(sample_times[0], segment_gps, rtol=0, atol=1e-6)
        assert np.allclose(np.diff(sample_times), 1 / SAMPLE_RATE, rtol=0, atol=5e-7)
        segment_ranges.append((float(segment_gps), float(segment_gps) + 1024))

        psd_frequencies, psd_values = signal.welch(
            clean_noise, fs=SAMPLE_RATE, nperseg=4 * SAMPLE_RATE
        )
        assert np.isfinite(psd_values).all() and np.all(psd_values > 0)
        candidate_starts = np.arange(
            edge_margin_samples,
            len(clean_noise) - edge_margin_samples - WINDOW_SAMPLES + 1,
            WINDOW_SAMPLES,
        )
        if len(candidate_starts) < 100:
            raise ValueError(f"Insufficient unique windows in {run} segment {segment_gps}")
        selected_starts = np.sort(rng.choice(candidate_starts, size=100, replace=False))
        for start_sample in selected_starts:
            center_sample = int(start_sample) + WINDOW_SAMPLES // 2
            buffer_start = center_sample - buffer_half_samples
            buffer_end = center_sample + buffer_half_samples
            noise_buffer = clean_noise[buffer_start:buffer_end]
            assert noise_buffer.size == 2 * buffer_half_samples
            fft_frequencies = np.fft.rfftfreq(noise_buffer.size, d=1 / SAMPLE_RATE)
            interpolated_psd = np.interp(fft_frequencies, psd_frequencies, psd_values)
            whitened_noise = np.fft.irfft(
                np.fft.rfft(noise_buffer) / np.sqrt(interpolated_psd), n=noise_buffer.size
            )
            filtered_noise = signal.filtfilt(filter_b, filter_a, whitened_noise)
            crop_start = int(start_sample) - buffer_start
            processed_window = filtered_noise[crop_start:crop_start + WINDOW_SAMPLES].copy()
            assert processed_window.shape == (WINDOW_SAMPLES,)
            assert np.isfinite(processed_window).all()
            processed_windows.append(processed_window)
            window_records.append({
                "run": run, "segment_gps": float(segment_gps),
                "start_sample": int(start_sample), "stop_sample": int(start_sample) + WINDOW_SAMPLES,
                "start_gps": float(segment_gps) + int(start_sample) / SAMPLE_RATE,
                "stop_gps": float(segment_gps) + (int(start_sample) + WINDOW_SAMPLES) / SAMPLE_RATE,
            })

    # Check source segments themselves do not overlap.
    assert all(left[1] <= right[0] for left, right in zip(segment_ranges, segment_ranges[1:]))
    # Shuffle only within this run; never mix observing runs.
    order = rng.permutation(len(processed_windows))
    negative_by_run[run] = np.stack(processed_windows)[order]
    negative_records_by_run[run] = [window_records[i] for i in order]
    segment_ranges_by_run[run] = segment_ranges
    assert negative_by_run[run].shape == (800, WINDOW_SAMPLES)
    print(run, "negative array:", negative_by_run[run].shape, "unique windows:", len(window_records))
    del noise_segments, clean_noise, sample_times, processed_windows

negative_train = negative_by_run["O3a"][:750]
negative_validation = negative_by_run["O3a"][750:]
negative_test = negative_by_run["O4a"]
negative_records = {
    "train": negative_records_by_run["O3a"][:750],
    "validation": negative_records_by_run["O3a"][750:],
    "test": negative_records_by_run["O4a"],
}
print("One negative window:", negative_train[0].shape)
print("First five processed samples:", negative_train[0, :5])

O3a negative array: (800, 16384) unique windows: 800
O4a negative array: (800, 16384) unique windows: 800
One negative window: (16384,)
First five processed samples: [15.72835251 24.42807847 32.19215115 37.37539786 38.58607577]


## 6. Attach binary labels and shuffle within each fixed split

Stack positives and negatives as rows. Create labels ourselves: one for each positive, zero for each negative. The same row permutation is applied to examples and labels. There is no cross-run `train_test_split`.

O3a negatives use different non-overlapping windows for training and validation. However, their long source segments and PSD estimates are shared, and the existing positive split may also share source noise. This is an illustrative validation set, not a fully independent segment-level holdout. We cannot establish disjoint positive/negative background intervals from the positive arrays alone.

In [6]:
classifier_splits = {}
shuffle_orders = {}
run_by_split = {"train": "O3a", "validation": "O3a", "test": "O4a"}
for split_index, (split, positives, negatives) in enumerate([
    ("train", positive_train, negative_train),
    ("validation", positive_validation, negative_validation),
    ("test", positive_test, negative_test),
]):
    assert positives.shape == negatives.shape
    examples = np.concatenate([positives, negatives], axis=0)
    labels = np.concatenate([np.ones(len(positives), dtype=np.int8), np.zeros(len(negatives), dtype=np.int8)])
    order = np.random.default_rng(SEED + 10 + split_index).permutation(len(labels))
    classifier_splits[split] = (examples[order], labels[order])
    shuffle_orders[split] = order

X_train_classifier, y_train_classifier = classifier_splits["train"]
X_validation_classifier, y_validation_classifier = classifier_splits["validation"]
X_test_classifier, y_test_classifier = classifier_splits["test"]

## 7. Validate shapes, values, labels, and run separation

Print every final array shape and both class counts. Check finite sample values, equal positive/negative shapes, run provenance, and non-overlapping negative windows. The checkpoint filenames and sidecars establish observing-run provenance; the positive arrays themselves do not carry per-row timestamps.

In [7]:
for split, expected_rows in [("train", 1500), ("validation", 100), ("test", 1600)]:
    examples, labels = classifier_splits[split]
    classes, counts = np.unique(labels, return_counts=True)
    print(f"{split}: X={examples.shape}, y={labels.shape}, run={run_by_split[split]}")
    print("  class counts:", dict(zip(classes.tolist(), counts.tolist())), "all finite:", np.isfinite(examples).all())
    assert examples.shape == (expected_rows, WINDOW_SAMPLES)
    assert labels.shape == (expected_rows,)
    assert np.array_equal(classes, [0, 1])
    assert np.array_equal(counts, [expected_rows // 2, expected_rows // 2])
    assert np.isfinite(examples).all()
    assert examples[labels == 0].shape == examples[labels == 1].shape
    assert {record["run"] for record in negative_records[split]} == {run_by_split[split]}

assert run_by_split == {"train": "O3a", "validation": "O3a", "test": "O4a"}
assert max(end for _, end in segment_ranges_by_run["O3a"]) < min(start for start, _ in segment_ranges_by_run["O4a"])
for run in ["O3a", "O4a"]:
    records = negative_records_by_run[run]
    identities = {(r["segment_gps"], r["start_sample"]) for r in records}
    assert len(identities) == 800
    intervals = sorted((r["start_gps"], r["stop_gps"]) for r in records)
    assert all(left[1] <= right[0] for left, right in zip(intervals, intervals[1:]))
print("Verified: O3a only for training/validation; O4a only for final testing.")
print("Verified: no repeated or overlapping final negative windows within either run.")

train: X=(1500, 16384), y=(1500,), run=O3a
  class counts: {0: 750, 1: 750} all finite: True
validation: X=(100, 16384), y=(100,), run=O3a
  class counts: {0: 50, 1: 50} all finite: True
test: X=(1600, 16384), y=(1600,), run=O4a
  class counts: {0: 800, 1: 800} all finite: True
Verified: O3a only for training/validation; O4a only for final testing.
Verified: no repeated or overlapping final negative windows within either run.


## 8. Save to a separate classification artifact

Save exactly the seven requested keys. Metadata records the source files, preprocessing, run split, random seeds, and each selected negative window. `shuffle_orders` maps each saved row back to its position in the positive-then-negative stack. Opening with `xb` creates a new file exclusively and refuses to overwrite an existing artifact. Rerunning this notebook stops if the output already exists.

In [8]:
metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "task": "binary simulated gravitational-wave event detection",
    "labels": {0: "noise only", 1: "simulated event present"},
    "positive_source": str(positive_path.relative_to(project_root)),
    "positive_keys": {"train": "X_train", "validation": "X_test_o3a", "test": "X_test_o4a"},
    "noise_sources": {run: str(path.relative_to(project_root)) for run, path in noise_paths.items()},
    "noise_source_metadata": noise_metadata,
    "run_by_split": run_by_split,
    "sample_rate_hz": SAMPLE_RATE, "window_samples": WINDOW_SAMPLES,
    "preprocessing": {
        "psd": "scipy.signal.welch on full original clean segment; default Hann window and 50% overlap",
        "psd_nperseg": 4 * SAMPLE_RATE,
        "whitening": "rfft / sqrt(interpolated PSD), then irfft; same normalization as existing pipeline",
        "bandpass_hz": [35, 350], "butterworth_order": 4, "filter": "filtfilt",
        "negative_context_seconds": 30, "crop_seconds": 4,
        "negative_waveform_injected": False, "positives_reprocessed": False,
    },
    "seed": SEED, "negative_run_seeds": {"O3a": SEED, "O4a": SEED + 1},
    "split_shuffle_seeds": {"train": SEED + 10, "validation": SEED + 11, "test": SEED + 12},
    "negative_window_records": negative_records, "shuffle_orders": shuffle_orders,
    "limitations": [
        "Event exclusion relies on acquisition-time catalog vetting, not a fresh catalog query.",
        "O3a training/validation share source segments and PSDs; positive background overlap is not recoverable from X arrays.",
        "Negative final windows are unique and non-overlapping; their processing context buffers can overlap.",
        "Balanced simulated-event classification is not a calibrated search for real events in continuous data.",
    ],
    "versions": {"numpy": np.__version__, "scipy": scipy.__version__, "sklearn": sklearn.__version__},
}
classification_dataset = {
    "X_train": X_train_classifier, "y_train": y_train_classifier,
    "X_validation": X_validation_classifier, "y_validation": y_validation_classifier,
    "X_test": X_test_classifier, "y_test": y_test_classifier,
    "metadata": metadata,
}
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("xb") as handle:
    pickle.dump(classification_dataset, handle, protocol=pickle.HIGHEST_PROTOCOL)
assert output_path.is_file() and output_path.stat().st_size > 0
print("Saved:", output_path)
print("Output exists:", output_path.exists(), "bytes:", output_path.stat().st_size)
print("Saved keys:", list(classification_dataset))

Saved: /Users/alilordifar/Library/CloudStorage/GoogleDrive-ali.lordifar@gmail.com/My Drive/Projects/ligo-parameter-inference/artifacts/classification/o3a_train_o4a_event_detection.pkl
Output exists: True bytes: 419549886
Saved keys: ['X_train', 'y_train', 'X_validation', 'y_validation', 'X_test', 'y_test', 'metadata']


## 9. Fit a simple raw-window logistic regression baseline

Each of the 16384 sample positions is a feature. There is **no PCA or feature extraction**. Standardization rescales each feature using O3a training statistics only; the classifier and scaler never fit on validation or test data. Logistic regression learns a linear decision rule on these scaled raw samples. We fix its settings here rather than selecting them using O4a.

This intentionally simple baseline can struggle with varying waveform phase and timing. A low score is still an informative result; it does not establish that signals are undetectable.

In [9]:
baseline = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, solver="liblinear", max_iter=2000, random_state=SEED),
)
with warnings.catch_warnings(record=True) as fit_warnings:
    warnings.simplefilter("always", ConvergenceWarning)
    baseline.fit(X_train_classifier, y_train_classifier)
for warning in fit_warnings:
    print("Fit warning:", str(warning.message))
print("Training features:", baseline.named_steps["logisticregression"].n_features_in_)
print("Solver iterations:", baseline.named_steps["logisticregression"].n_iter_)
assert baseline.named_steps["logisticregression"].n_features_in_ == WINDOW_SAMPLES

Training features: 16384
Solver iterations: [9]


## 10. Evaluate on O3a validation, then O4a final testing

Accuracy is the fraction of correct predictions. Precision asks how many predicted events really contain an injection. Recall asks how many injected events are found. The confusion matrix uses rows = true class and columns = predicted class, in order `[0, 1]`: `[[TN, FP], [FN, TP]]`. ROC-AUC uses event probabilities across thresholds; both classes are present, so it is defined here. Classification uses the default probability threshold of 0.5.

The **O4a result measures cross-run generalization**: the classifier learns from O3a noise and events, then is evaluated on different O4a noise and events. Do not tune the model or threshold using this final-test result. These balanced simulated datasets do not measure real-world event rates or false alarms per observing hour.

In [10]:
metrics_by_split = {}
for title, split, examples, labels in [
    ("O3a validation", "validation", X_validation_classifier, y_validation_classifier),
    ("O4a final test — cross-run generalization", "test", X_test_classifier, y_test_classifier),
]:
    predicted_labels = baseline.predict(examples)
    event_probabilities = baseline.predict_proba(examples)[:, 1]
    assert np.isfinite(event_probabilities).all()
    metrics = {
        "accuracy": accuracy_score(labels, predicted_labels),
        "precision": precision_score(labels, predicted_labels, zero_division=0),
        "recall": recall_score(labels, predicted_labels, zero_division=0),
        "roc_auc": roc_auc_score(labels, event_probabilities),
    }
    metrics_by_split[split] = metrics
    print("\n" + title)
    for name, value in metrics.items():
        print(f"  {name}: {value:.4f}")
    print("  Confusion matrix [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(labels, predicted_labels, labels=[0, 1]))
print("\nDataset file exists:", output_path.is_file())


O3a validation
  accuracy: 0.5100
  precision: 0.5116
  recall: 0.4400
  roc_auc: 0.5552
  Confusion matrix [[TN, FP], [FN, TP]]:
[[29 21]
 [28 22]]

O4a final test — cross-run generalization
  accuracy: 0.5312
  precision: 0.5403
  recall: 0.4188
  roc_auc: 0.5422
  Confusion matrix [[TN, FP], [FN, TP]]:
[[515 285]
 [465 335]]

Dataset file exists: True
